In [ ]:
# import pandas as pd

# train_df = pd.read_json("data/processed/sft_dataset/train.jsonl", lines=True)
# val_df = pd.read_json("data/processed/sft_dataset/validation.jsonl", lines=True)
# # test_df = pd.read_json("data/processed/sft_dataset/test.jsonl", lines=True)

# print("Train:", train_df.shape)
# print("Validation:", val_df.shape)
# print("Test:", test_df.shape)

# display(train_df.head())

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving test.jsonl to test (1).jsonl
Saving train.jsonl to train (1).jsonl
Saving validation.jsonl to validation (1).jsonl


In [1]:
import pandas as pd

train_df = pd.read_json(
    "data/processed/sft_dataset/train.jsonl",
    lines=True
)

val_df = pd.read_json(
    "data/processed/sft_dataset/validation.jsonl",
    lines=True
)

test_df = pd.read_json(
    "data/processed/sft_dataset/test.jsonl",
    lines=True
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (3901, 2)
Validation: (488, 2)
Test: (488, 2)


In [2]:
from transformers import AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print("Tokenizer loaded successfully!")
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)

Tokenizer loaded successfully!
EOS token: <|im_end|>
EOS token ID: 151645


In [3]:
print("Prompt:")
print(train_df.iloc[0]["prompt"][:500])

print("\nCompletion:")
print(train_df.iloc[0]["completion"][:500])

Prompt:
<|im_start|>system
You are an expert resume parser. Extract structured information from resumes and return ONLY valid JSON. Do not include explanations or extra text.<|im_end|>
<|im_start|>user
Resume:
LEAD ACCOUNTANT
Highlights
QuickBooks, Peachtree, In-house Accounting Systems and Financial Reporting, MAS 90, FAR, WAWF, Great Plains, Integration
Manager, FRx reporting.
Hyperion Workspace, Planning, Enterprise, Essbase, Schedules and Reports, Crystal Reports, MS Office with strong Excel skill
F

Completion:
{"current_title": "Lead Accountant", "previous_titles": ["Senior Accountant", "Staff Accountant"], "current_company": null, "previous_companies": [], "years_experience": 20.0, "seniority": "lead", "primary_domain": "Accounting/Finance", "industries": [], "core_skills": ["Financial Reporting", "Budgeting", "Forecasting/Trend Analysis"], "secondary_skills": ["Internal Controls / Sarbanes-Oxley compliance", "Cash-Flow Analysis", "Audit Coordination"], "tools": ["QuickBooks", "

In [4]:
def count_tokens(row):
    prompt_tokens = tokenizer(
        row["prompt"],
        add_special_tokens=False
    )["input_ids"]

    completion_tokens = tokenizer(
        row["completion"],
        add_special_tokens=False
    )["input_ids"]

    return len(prompt_tokens) + len(completion_tokens)


train_df["token_length"] = train_df.apply(count_tokens, axis=1)

print(train_df["token_length"].describe())
print("Maximum tokens:", train_df["token_length"].max())

count    3901.000000
mean     1673.372981
std      1000.616665
min       190.000000
25%      1083.000000
50%      1425.000000
75%      1883.000000
max      9388.000000
Name: token_length, dtype: float64
Maximum tokens: 9388


In [5]:
limits = [2048, 3072, 4096, 6144, 8192]

for limit in limits:
    count = (train_df["token_length"] > limit).sum()

    print(
        f"Above {limit}: {count} "
        f"({count / len(train_df) * 100:.2f}%)"
    )

Above 2048: 798 (20.46%)
Above 3072: 352 (9.02%)
Above 4096: 169 (4.33%)
Above 6144: 12 (0.31%)
Above 8192: 2 (0.05%)


In [6]:
train_df["prompt_length"] = train_df["prompt"].apply(
    lambda x: len(tokenizer(x, add_special_tokens=False)["input_ids"])
)

train_df["completion_length"] = train_df["completion"].apply(
    lambda x: len(tokenizer(x, add_special_tokens=False)["input_ids"])
)

print("PROMPT LENGTH")
print(train_df["prompt_length"].describe())

print("\nCOMPLETION LENGTH")
print(train_df["completion_length"].describe())

PROMPT LENGTH
count    3901.000000
mean     1400.843117
std       986.753119
min        75.000000
25%       802.000000
50%      1155.000000
75%      1603.000000
max      9100.000000
Name: prompt_length, dtype: float64

COMPLETION LENGTH
count    3901.000000
mean      272.529864
std        33.860815
min       115.000000
25%       249.000000
50%       271.000000
75%       294.000000
max       401.000000
Name: completion_length, dtype: float64


In [7]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    gpu = torch.cuda.get_device_properties(0)

    print(
        "VRAM:",
        round(gpu.total_memory / (1024**3), 2),
        "GB"
    )
else:
    print("GPU not available. Enable GPU in Runtime settings.")

GPU not available. Enable GPU in Runtime settings.


In [8]:
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import prepare_model_for_kbit_training

In [9]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [10]:
import bitsandbytes as bnb
import torch

from transformers.utils import is_bitsandbytes_available

print("BitsAndBytes:", bnb.__version__)
print("GPU:", torch.cuda.get_device_name(0))

# Clear cached availability check
is_bitsandbytes_available.cache_clear()

print(
    "BitsAndBytes detected:",
    is_bitsandbytes_available()
)

ModuleNotFoundError: No module named 'bitsandbytes'

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

In [ ]:
model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [ ]:
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(name)
        break

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight


In [ ]:
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["prompt", "completion"]].drop(columns=[], errors="ignore"),
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["prompt", "completion"]],
    preserve_index=False
)

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 3901
})
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 488
})


In [ ]:
MAX_LENGTH = 3072

def fit_example(example):
    prompt = example["prompt"]
    completion = example["completion"]

    prompt_ids = tokenizer(
        prompt, add_special_tokens=False
    )["input_ids"]

    completion_ids = tokenizer(
        completion, add_special_tokens=False
    )["input_ids"]

    suffix = "<|im_end|>\n<|im_start|>assistant\n"
    suffix_ids = tokenizer(
        suffix, add_special_tokens=False
    )["input_ids"]

    if prompt_ids[-len(suffix_ids):] != suffix_ids:
        raise ValueError("Assistant generation marker is missing.")

    prompt_budget = MAX_LENGTH - len(completion_ids)

    if prompt_budget < len(suffix_ids) + 1:
        raise ValueError("Completion is too long.")

    if len(prompt_ids) > prompt_budget:
        body_ids = prompt_ids[:-len(suffix_ids)]
        prompt_ids = (
            body_ids[:prompt_budget - len(suffix_ids)]
            + suffix_ids
        )
        prompt = tokenizer.decode(
            prompt_ids, skip_special_tokens=False
        )

    return {"prompt": prompt, "completion": completion}


train_dataset = train_dataset.map(fit_example)
val_dataset = val_dataset.map(fit_example)

Map:   0%|          | 0/3901 [00:00<?, ? examples/s]

Map:   0%|          | 0/488 [00:00<?, ? examples/s]

In [ ]:
def total_length(example):
    return (
        len(tokenizer(example["prompt"], add_special_tokens=False)["input_ids"])
        + len(tokenizer(example["completion"], add_special_tokens=False)["input_ids"])
    )

print("Maximum train length:", max(total_length(x) for x in train_dataset))
print("Maximum validation length:", max(total_length(x) for x in val_dataset))

Maximum train length: 3072
Maximum validation length: 3072


In [ ]:
%pip install -q trl==1.17.1
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./resumeiq-qwen-lora",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    warmup_steps=15,
    lr_scheduler_type="cosine",

    fp16=False,
    bf16=False,

    optim="paged_adamw_8bit",

    max_length=3072,
    completion_only_loss=True,
    packing=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    eval_strategy="steps",
    eval_steps=200,

    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,

    logging_steps=10,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none"
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer
)

for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

print("Trainer ready!")

Adding EOS to train dataset:   0%|          | 0/3901 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3901 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3901 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3901 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3901 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/488 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/488 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/488 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/488 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/488 [00:00<?, ? examples/s]

Trainer ready!


In [ ]:
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

print("Trainable parameters converted!")

Trainable parameters converted!


In [ ]:
from collections import Counter

print(Counter(
    str(p.dtype)
    for p in trainer.model.parameters()
    if p.requires_grad
))
training_args.fp16 = False
training_args.bf16 = False

print("FP16:", training_args.fp16)
print("BF16:", training_args.bf16)

Counter({'torch.float32': 504})
FP16: False
BF16: False


In [ ]:
import torch

torch.cuda.empty_cache()

training_result = trainer.train()

print("Training completed!")
print(training_result)

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
200,0.495069,0.489577,0.497403,0.860342,1231496.000000
400,0.490796,0.509790,0.503890,0.857883,2489364.000000
600,0.518213,0.543166,0.579747,0.854180,3768988.000000
800,0.584392,0.526459,0.544781,0.858225,5022291.000000
976,0.540047,0.522147,0.532013,0.859587,6116464.000000


Training completed!
TrainOutput(global_step=976, training_loss=0.5431060703074346, metrics={'train_runtime': 13933.9143, 'train_samples_per_second': 0.28, 'train_steps_per_second': 0.07, 'total_flos': 1.0292934841348915e+17, 'train_loss': 0.5431060703074346, 'epoch': 1.0})


In [ ]:
%pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("GPU is not enabled!")

In [ ]:
SAVE_DIR = "/content/resumeiq-finetuned"

trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Model saved successfully!")

Model saved successfully!


In [ ]:
import shutil

zip_path = shutil.make_archive(
    "/content/resumeiq-finetuned",
    "zip",
    SAVE_DIR
)

print("ZIP created:", zip_path)

ZIP created: /content/resumeiq-finetuned.zip


In [ ]:
from google.colab import files

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
model = trainer.model
model.eval()

print("Fine-tuned model ready!")

NameError: name 'trainer' is not defined

In [2]:
import os
import json
from pathlib import Path

# Project directory
PROJECT_DIR = Path.cwd()

print("Project directory:", PROJECT_DIR)

# Check adapter files
print("\nAdapter files:")

for filename in [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json"
]:
    path = PROJECT_DIR / filename
    print(filename, ":", path.exists())

# Check validation dataset
VALIDATION_PATH = PROJECT_DIR / "data" / "processed" / "validation.jsonl"

print("\nValidation file exists:", VALIDATION_PATH.exists())

Project directory: c:\Users\vishn\OneDrive\Desktop\LLM

Adapter files:
adapter_config.json : True
adapter_model.safetensors : True
tokenizer.json : True
tokenizer_config.json : True

Validation file exists: False


In [3]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

print("Current directory:", PROJECT_DIR)

print("\nFiles and folders:")

for item in PROJECT_DIR.iterdir():
    print(item.name, "->", "Folder" if item.is_dir() else "File")

Current directory: c:\Users\vishn\OneDrive\Desktop\LLM

Files and folders:
02_data_preprocessing.ipynb -> File
adapter_config.json -> File
adapter_model.safetensors -> File
chat_template.jinja -> File
data -> Folder
QLoRA_Fine_Tuning (2).ipynb -> File
README.md -> File
tokenizer.json -> File
tokenizer_config.json -> File


In [4]:
from pathlib import Path

DATA_DIR = Path.cwd() / "data"

print("Data directory:", DATA_DIR)

for path in DATA_DIR.rglob("*"):
    if path.is_file():
        print("FILE:", path.relative_to(Path.cwd()))
    elif path.is_dir():
        print("FOLDER:", path.relative_to(Path.cwd()))

Data directory: c:\Users\vishn\OneDrive\Desktop\LLM\data
FOLDER: data\processed
FOLDER: data\processed\sft_dataset
FILE: data\processed\sft_dataset\test.jsonl
FILE: data\processed\sft_dataset\train.jsonl
FILE: data\processed\sft_dataset\validation.jsonl


In [5]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

VALIDATION_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "sft_dataset"
    / "validation.jsonl"
)

print("Validation file exists:", VALIDATION_PATH.exists())
print("Validation path:", VALIDATION_PATH)

Validation file exists: True
Validation path: c:\Users\vishn\OneDrive\Desktop\LLM\data\processed\sft_dataset\validation.jsonl


In [6]:
import json

with open(VALIDATION_PATH, "r", encoding="utf-8") as f:
    validation_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Total validation records:", len(validation_records))

print("\nKeys:", validation_records[0].keys())

print("\nPrompt:")
print(validation_records[0]["prompt"][:1000])

print("\nExpected completion:")
print(validation_records[0]["completion"])

Total validation records: 488

Keys: dict_keys(['prompt', 'completion'])

Prompt:
<|im_start|>system
You are an expert resume parser. Extract structured information from resumes and return ONLY valid JSON. Do not include explanations or extra text.<|im_end|>
<|im_start|>user
Resume:
FREELANCE PRODUCTION MANAGER - MEN'S & WOMEN'S WOVENS/KNITS
Summary
Creative Development and Production Manager with a strong background in Men's and Women's Wovens, Denim, Knits, and Graphics. Passion
to expand upon current skill sets and areas of expertise. Sets and maintains high personal and professional standards. Creative strategic thinker;
strong ability to quickly identify, analyze, and resolve challenges to achieve the most effective outcome.
Highlights
Fabric and trim vendor relationsExtensive print knowledge
Strong cost negotiation skills Wash development
Superior communication skills Apparel construction
WebPDM and PLM Experienced in traveling to Asia factories
Experience
Freelance Production Ma

In [7]:
for i, record in enumerate(validation_records[:5]):
    assert "prompt" in record, f"Missing prompt in record {i}"
    assert "completion" in record, f"Missing completion in record {i}"
    assert isinstance(record["prompt"], str)
    assert isinstance(record["completion"], str)

print("First 5 records passed structural validation.")

First 5 records passed structural validation.


In [11]:
import pandas as pd
import json

test_df = pd.read_json(
    "data/processed/sft_dataset/test.jsonl",
    lines=True
)

print("Dataset shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

print("\nFirst sample:")
print(json.dumps(test_df.iloc[0].to_dict(), indent=4))

print("\nMissing values:")
print(test_df.isnull().sum())

Dataset shape: (488, 2)
Columns: ['prompt', 'completion']

First sample:
{
    "prompt": "<|im_start|>system\nYou are an expert resume parser. Extract structured information from resumes and return ONLY valid JSON. Do not include explanations or extra text.<|im_end|>\n<|im_start|>user\nResume:\nIT Desktop Support Manager <span class=\"hl\">IT</span> Desktop Support <span class=\"hl\">Manager</span> IT Desktop Support Manager Las Vegas, NV Authorized to work in the US for any employer Work Experience IT Desktop Support Manager Gala Coral/Ladbrokes Coral - Jerusalem, IL 2016 to 2017  Managed suppliers and IT inventory  Managed tickets between various IT departments to complete goals  Troubleshooting on Macs  Technical Documentation for corporate  Connected Macs to Active directory  Set static IP on DHCP server  Manage tickets in JIRA and Service Now  Worked with DevOps to setup testing environments  Worked with Project and Product Managers  Worked with UK and Gibraltar Windows and Networ

In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_PATH ="/content/resumeiq-finetuned"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model.eval()

print("Model loaded successfully!")
print("Device:", next(model.parameters()).device)

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/resumeiq-finetuned'. Use `repo_type` argument if needed.

In [15]:
import sys

!{sys.executable} -m pip install accelerate

In [17]:
import sys
import accelerate

print("Python:", sys.executable)
print("Accelerate:", accelerate.__version__)
print("Location:", accelerate.__file__)

Python: c:\Users\vishn\anaconda3\envs\ann_env\python.exe
Accelerate: 1.15.0
Location: c:\Users\vishn\anaconda3\envs\ann_env\Lib\site-packages\accelerate\__init__.py


In [11]:

import torch
import json

def predict_resume(resume_text):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert resume parser. "
                "Extract structured information from resumes "
                "and return ONLY valid JSON. "
                "Do not include explanations or extra text."
            )
        },
        {
            "role": "user",
            "content": f"Resume:\n{resume_text}"
        }
    ]

    # Apply the same Qwen chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract only newly generated tokens
    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response


In [12]:

resume_1 = """
John Smith
Software Engineer

Email: john@example.com

Experience:
Software Engineer at ABC Technologies
2022 - 2025

Developed REST APIs using Python and Flask.
Worked with MySQL and AWS.

Skills:
Python, Java, Flask, Django, MySQL, AWS

Education:
B.Tech Computer Science, XYZ University, 2022
"""


resume_2 = """
Priya Sharma
Computer Science Graduate

Education:
B.Tech Computer Science, ABC University, 2025

Skills:
Python, Machine Learning, SQL, Java, React

Projects:
Loan Approval Prediction:
Developed a classification model using Scikit-learn.

Student Management System:
Built using Flask and MySQL.

Certifications:
Python Programming
Machine Learning Fundamentals
"""


resume_3 = """
Alex Kumar

Professional Summary:
Motivated developer interested in backend development.

Technical Skills:
Python, FastAPI, PostgreSQL, Docker

Projects:
AI-powered Resume Screening System.
Developed a resume parser using NLP.

Education:
Bachelor of Technology

No contact information provided.
No employment history provided.
"""


test_resumes = {
    "Experienced Candidate": resume_1,
    "Fresher Candidate": resume_2,
    "Incomplete Resume": resume_3
}


In [13]:

predictions = {}

for name, resume in test_resumes.items():

    print("\n", "=" * 60)
    print(name)
    print("=" * 60)

    response = predict_resume(resume)

    predictions[name] = response

    print(response)

    # Check JSON validity
    try:
        parsed = json.loads(response)
        print("\nJSON VALID: True")

    except json.JSONDecodeError as e:
        print("\nJSON VALID: False")
        print("Error:", e)



Experienced Candidate


NameError: name 'model' is not defined